[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/08_rmsnorm.ipynb)

# 🟡 中级：实现 RMSNorm

实现 **Root Mean Square Layer Normalization**（均方根层归一化）—— LLaMA、Gemma 等模型所使用的归一化方法。

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot w, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}$$

### 函数签名
```python
def rms_norm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # 在最后一个维度上进行归一化。不减均值（与 LayerNorm 不同）。
```

### 规则
- **禁止** 使用任何内置的归一化层（如 `nn.LayerNorm`、`F.rms_norm` 等）
- 必须在 `dim=-1` 上进行归一化
- 必须支持自动求导（autograd）

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

RMS(x) = sqrt( mean(x^2) + eps )
RMSNorm(x) = x / RMS(x) * w

In [ ]:
def rms_norm(x, weight, eps=1e-6):
    '''
    x: 输入张量
    weight: 最后权重
    eps: 极小正数
    '''
    # x shape: (..., d), weight shape: (d,)
    
    # 1. 计算 RMS (Root Mean Square)
    # 在最后一个维度上计算平方的均值
    rms = x.pow(2).mean(dim=-1, keepdim=True)
    # 加上 epsilon 防止除零，然后开平方
    rms = torch.sqrt(rms + eps)
    
    # 2. 归一化
    # 公式: x_normalized = x / RMS(x)
    x_normalized = x / rms
    
    # 3. 应用权重 (缩放)
    # 公式: out = weight * x_normalized
    out = weight * x_normalized
    
    return out

In [ ]:
# 🧪 调试测试
x = torch.randn(2, 8)
w = torch.ones(8)
out = rms_norm(x, w)
print("输出形状:", out.shape)
print("输出的 RMS 值:", out.pow(2).mean(dim=-1).sqrt())  # 应该接近 1

In [ ]:
from torch_judge import check
check('rmsnorm')